# Phase 4b — Hypothesis Tests + Classifier (n = 127, expanded sample)Same structure as Phase 4a, pointed at the expanded 127-record dataset (the original 75plus 52 newly collected, misdirection-domain records). This is the **replication** sideof the paper's central comparison — every number here should be compared directlyagainst Phase 4a's equivalent.Fully independent of Phase 4a: rebuilds every feature from scratch, no shared state.

In [ ]:
import numpy as npimport jsonimport osfrom collections import Counterimport matplotlib.pyplot as pltimport seaborn as snsfrom statsmodels.stats.multitest import multipletestsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_scorenp.random.seed(42)NPZ_PATH  = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_attention_v3.npz'META_PATH = '/kaggle/input/notebooks/gunjalgupta/phase3-v2/pivot_research/phase3_metadata_v3.jsonl'OUTPUT_DIR = '/kaggle/working/pivot_research'os.makedirs(OUTPUT_DIR, exist_ok=True)N_LAYERS, N_HEADS, N_QUERIES = 28, 12, 30EARLY_KEY_RANGE = 50EARLY_LAYERS = list(range(0, N_LAYERS // 3))LATE_LAYERS  = list(range(2 * N_LAYERS // 3, N_LAYERS))

## Load dataSame defensive pattern as Phase 4a: assert the sample size and label counts immediately,so a wrong path fails here rather than downstream.

In [ ]:
npz = np.load(NPZ_PATH)with open(META_PATH) as f:    metadata = [json.loads(l) for l in f if l.strip()]n_records = len(metadata)weights = np.stack([npz[f'weights_{i:03d}'] for i in range(n_records)], axis=0)outputs = np.stack([npz[f'outputs_{i:03d}'] for i in range(n_records)], axis=0)labels = np.array([m['label'] for m in metadata])categories = np.array([m['category'] for m in metadata])y = (labels == 'correction').astype(int)valid_key_starts = np.array([m['valid_key_start'] for m in metadata])prompt_ids = np.array([m['prompt_id'] for m in metadata])   # kept for Phase 4c cross-checkscorr_mask, halluc_mask = y == 1, y == 0print(f"n={n_records}: correction={corr_mask.sum()}, hallucination_pivot={halluc_mask.sum()}")assert n_records == 127 and corr_mask.sum() == 22, \    f"Expected n=127, 22 correction -- got n={n_records}, {corr_mask.sum()} correction. Check NPZ_PATH."print("Category distribution:", dict(Counter(categories)))

## Shared utilitiesIdentical to Phase 4a — permutation test + FDR correction, applied per head.

In [ ]:
def permutation_test(group1, group2, n_perm=2000, seed=42):    rng = np.random.default_rng(seed)    observed = group1.mean() - group2.mean()    combined = np.concatenate([group1, group2])    n1 = len(group1)    diffs = np.zeros(n_perm)    for i in range(n_perm):        rng.shuffle(combined)        diffs[i] = combined[:n1].mean() - combined[n1:].mean()    return observed, (np.abs(diffs) >= np.abs(observed)).mean()def run_per_head_test(features):    p_values = np.zeros((N_LAYERS, N_HEADS))    effect_sizes = np.zeros((N_LAYERS, N_HEADS))    for L in range(N_LAYERS):        for H in range(N_HEADS):            diff, p = permutation_test(features[corr_mask, L, H], features[halluc_mask, L, H])            p_values[L, H] = p            effect_sizes[L, H] = diff    reject, adj_p, _, _ = multipletests(p_values.flatten(), alpha=0.05, method='fdr_bh')    adj_p = np.asarray(adj_p).reshape(N_LAYERS, N_HEADS)    reject = np.asarray(reject).reshape(N_LAYERS, N_HEADS)    return p_values, effect_sizes, adj_p, reject

## H1 — Early-Context Attention SpikeSame definition as Phase 4a — see that notebook for the full explanation. Feature:attention mass on the first 50 real keys, averaged over the 30-query window.

In [ ]:
def compute_h1(weights, valid_key_starts):    h1 = np.zeros((n_records, N_LAYERS, N_HEADS))    for i in range(n_records):        v = valid_key_starts[i]        early_end = min(v + EARLY_KEY_RANGE, weights.shape[-1])        h1[i] = weights[i, :, :, :, v:early_end].sum(axis=-1).mean(axis=-1)    return h1h1_features = compute_h1(weights, valid_key_starts)p_values, effect_sizes, adj_p, reject = run_per_head_test(h1_features)print(f"H1: {reject.sum()}/336 significant (FDR); best raw p = {p_values.min():.4f}")

## H2 — Consistency-Head ActivationSame definition as Phase 4a. Feature: L2 norm of head output, averaged over the query window.

In [ ]:
h2_features = np.linalg.norm(outputs, axis=-1).mean(axis=2)p_values_h2, effect_sizes_h2, adj_p_h2, reject_h2 = run_per_head_test(h2_features)print(f"H2: {reject_h2.sum()}/336 significant (FDR); best adj p = {adj_p_h2.min():.4f}")

## H3 — Attention Entropy (normalized)Same corrected, normalized definition as Phase 4a — see that notebook for the fullrationale. The assertion below checks against n=127's expected rate specifically(~10.1%, slightly different from n=75's ~10.7% due to sample composition).

In [ ]:
def entropy_normalized(row, valid_start):    real = row[valid_start:]    real = real[real > 1e-10]    K_q = len(real)    if K_q <= 1 or real.sum() == 0:        return 0.0    p = real / real.sum()    raw = -np.sum(p * np.log(p))    return raw / np.log(K_q)def compute_h3(weights, valid_key_starts):    h3 = np.zeros((n_records, N_LAYERS, N_HEADS))    for i in range(n_records):        v = valid_key_starts[i]        for L in range(N_LAYERS):            for H in range(N_HEADS):                ents = [entropy_normalized(weights[i, L, H, q, :], v) for q in range(N_QUERIES)]                h3[i, L, H] = np.mean(ents)    return h3h3_features = compute_h3(weights, valid_key_starts)p_values_h3, effect_sizes_h3, adj_p_h3, reject_h3 = run_per_head_test(h3_features)pct_below_05 = (p_values_h3 < 0.05).mean()print(f"H3: {reject_h3.sum()}/336 significant (FDR); best adj p = {adj_p_h3.min():.4f}; "      f"{pct_below_05:.1%} raw p<0.05 (expect ~10.1% at n=127)")assert 0.07 < pct_below_05 < 0.14, \    f"H3 rate {pct_below_05:.1%} outside expected ~10.1% range -- check entropy_normalized is being used"

## H4 — Late/Early Layer Magnitude RatioSame definition as Phase 4a. This is the hypothesis closest to significance at n=75(p=0.057) — watch whether it weakens here, as the paper's central pattern predicts.

In [ ]:
early_mag = h2_features[:, EARLY_LAYERS, :].mean(axis=(1, 2))late_mag  = h2_features[:, LATE_LAYERS, :].mean(axis=(1, 2))h4_features = late_mag / (early_mag + 1e-10)diff_h4, p_h4 = permutation_test(h4_features[halluc_mask], h4_features[corr_mask], n_perm=10000)print(f"H4: diff={diff_h4:+.4f}, p={p_h4:.4f}")

## Figure D1 — P-value diagnosticsHistograms and QQ plots for H1–H3's per-head p-values, testing whether there's anysystematic enrichment of small p-values even below the individual-head FDR-significancethreshold. This is the figure referenced as Figure D1 in the paper's Appendix D.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))hypothesis_data = [    ('H1: Early-Context Spike', p_values.flatten()),    ('H2: Consistency Activation', p_values_h2.flatten()),    ('H3: Attention Entropy (normalized)', p_values_h3.flatten()),]for col, (title, pvals) in enumerate(hypothesis_data):    ax = axes[0, col]    n_bins = 20    ax.hist(pvals, bins=n_bins, range=(0, 1), color='#4C72B0', alpha=0.75, edgecolor='#2E4A78')    expected_per_bin = len(pvals) / n_bins    ax.axhline(expected_per_bin, color='#C44E52', linestyle='--', linewidth=1.5,               label=f'expected under null ({expected_per_bin:.1f}/bin)')    ax.set_title(title, fontsize=10.5, fontweight='bold')    ax.set_xlabel('p-value'); ax.set_ylabel('count' if col == 0 else '')    ax.legend(fontsize=7.5, loc='upper right')    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)    ax2 = axes[1, col]    n = len(pvals)    observed = -np.log10(np.sort(pvals))    expected = -np.log10(np.arange(1, n + 1) / (n + 1))    max_val = max(observed.max(), expected.max()) * 1.05    ax2.scatter(expected, observed, s=12, color='#4C72B0', alpha=0.6, edgecolor='none')    ax2.plot([0, max_val], [0, max_val], color='#C44E52', linestyle='--', linewidth=1.5, label='y = x (null)')    ax2.set_xlabel('expected -log10(p)'); ax2.set_ylabel('observed -log10(p)' if col == 0 else '')    ax2.set_xlim(0, max_val); ax2.set_ylim(0, max_val)    ax2.legend(fontsize=7.5, loc='upper left')    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)fig.suptitle('P-value diagnostics across all 336 attention heads (H1-H3), n=127',              fontsize=12.5, fontweight='bold', y=1.00)plt.tight_layout()plt.savefig(f'{OUTPUT_DIR}/fig_D1_pvalue_diagnostics_n127.png', dpi=150, bbox_inches='tight', facecolor='white')plt.show()print(f"Saved Figure D1 to {OUTPUT_DIR}/fig_D1_pvalue_diagnostics_n127.png")

## Combined classifierSame 1,009-feature, C=0.01, instance-level 5-fold CV setup as Phase 4a. This is the**pooled n=127** number — original and follow-up records can appear across foldstogether here (Phase 4c tests what happens when that's not allowed).

In [ ]:
X = np.concatenate([    h1_features.reshape(n_records, -1),    h2_features.reshape(n_records, -1),    h3_features.reshape(n_records, -1),    h4_features.reshape(n_records, 1),], axis=1)print(f"Feature matrix: {X.shape}")skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)all_preds, all_probas = np.zeros(n_records), np.zeros(n_records)fold_results = []for fold_idx, (tr, te) in enumerate(skf.split(X, y)):    sc = StandardScaler()    Xtr, Xte = sc.fit_transform(X[tr]), sc.transform(X[te])    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)    clf.fit(Xtr, y[tr])    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]    all_preds[te], all_probas[te] = preds, probas    ba = balanced_accuracy_score(y[te], preds)    fold_results.append(ba)    print(f"  Fold {fold_idx}: bal_acc={ba:.3f}  (n_correction={int((y[te]==1).sum())})")print(f"\nOverall: bal_acc={balanced_accuracy_score(y, all_preds):.3f}, "      f"F1={f1_score(y, all_preds, zero_division=0):.3f}, "      f"ROC-AUC={roc_auc_score(y, all_probas):.3f}")print(f"Fold mean±SD: {np.mean(fold_results):.3f} ± {np.std(fold_results):.3f}")

## Cross-domain ablationSame as Phase 4a. Compare the math result here against Phase 4a's — this is thepaper's sharpest single illustration of non-replication.

In [ ]:
print("Cross-domain ablation (train on 2 categories, test on 3rd):")for held_out in ['math', 'logic', 'misdirection']:    test_mask = categories == held_out    train_mask = ~test_mask    sc = StandardScaler()    Xtr, Xte = sc.fit_transform(X[train_mask]), sc.transform(X[test_mask])    clf = LogisticRegression(class_weight='balanced', max_iter=2000, C=0.01, random_state=42)    clf.fit(Xtr, y[train_mask])    preds, probas = clf.predict(Xte), clf.predict_proba(Xte)[:, 1]    ba = balanced_accuracy_score(y[test_mask], preds)    auc = roc_auc_score(y[test_mask], probas)    print(f"  {held_out}: n={test_mask.sum()}, bal_acc={ba:.3f}, AUC={auc:.3f}")

## Save results

In [ ]:
results = {    'n_records': n_records,    'h1': {'n_sig': int(reject.sum()), 'best_adj_p': float(adj_p.min())},    'h2': {'n_sig': int(reject_h2.sum()), 'best_adj_p': float(adj_p_h2.min())},    'h3': {'n_sig': int(reject_h3.sum()), 'best_adj_p': float(adj_p_h3.min())},    'h4': {'p_value': float(p_h4), 'diff': float(diff_h4)},    'classifier': {        'balanced_accuracy': float(balanced_accuracy_score(y, all_preds)),        'f1': float(f1_score(y, all_preds, zero_division=0)),        'roc_auc': float(roc_auc_score(y, all_probas)),    },}with open(f'{OUTPUT_DIR}/phase4b_n127_results.json', 'w') as f:    json.dump(results, f, indent=2)print(f"Saved to {OUTPUT_DIR}/phase4b_n127_results.json")